# 04 — LLM Diagnostic Layer

Turns a segmentation prediction into a **diagnostic note** (likely cause + severity + action),
grounded in deterministic measurements + the paper's real T1 cause vocabulary. This is the
project's most distinctive contribution: the paper's text is *descriptive*; this is *diagnostic*
(cause + action), which the paper explicitly flagged as future work.

**Pipeline:** predicted mask → [deterministic] T3-style attributes → [retrieve] T1 cause →
[LLM, constrained] diagnostic JSON + prose. The LLM never sees the image and cannot invent causes.

**Setup:** set your API key (cell 2). Runs anywhere with internet — no GPU needed for the LLM part
(you do need the trained model + a few test images for the segmentation step).

In [ ]:
# deps
!pip install -q google-generativeai segmentation-models-pytorch albumentations
import os, sys
# ensure repo root on path (adjust if running outside the repo)
sys.path.insert(0, os.getcwd())

## 1. API key
On Kaggle: Add-ons → Secrets → add `GEMINI_API_KEY`, then load it.
Locally: set the env var before launching. **Never hard-code the key in the notebook.**
Get a free key (no credit card) at aistudio.google.com → Get API key.

In [ ]:
# Kaggle:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    print('key loaded from Kaggle secret')
except Exception:
    assert os.environ.get('GEMINI_API_KEY'), 'set GEMINI_API_KEY env var'
    print('key from env')
PROVIDER = 'gemini'   # free tier via Google AI Studio (aistudio.google.com)

## 2. Load T1 causes + one test image, predict a mask

In [ ]:
import numpy as np, cv2, torch
from src.llm.diagnose import load_t1_causes
from src.llm.attributes import extract_attributes
from src.segmentation.model import build_model, enable_mc_dropout

DATA_ROOT = 'sdx_data'        # adjust to your data path
CKPT      = 'models/unet_dropout.pt'   # the deployed dropout-U-Net from notebook 03

t1 = load_t1_causes(f'{DATA_ROOT}/class_descriptions.json')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model('unet', 'resnet34', None, dropout=0.2)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.to(device).eval()

def predict_with_conf(img_path, mc=30):
    g = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (256,256))
    x = np.stack([g,g,g],0)[None].astype(np.float32)/255.0
    mean = np.array([0.485,0.456,0.406])[None,:,None,None]
    std  = np.array([0.229,0.224,0.225])[None,:,None,None]
    x = torch.tensor((x-mean)/std, dtype=torch.float32).to(device)
    enable_mc_dropout(model)
    with torch.no_grad():
        ps = torch.stack([torch.sigmoid(model(x)) for _ in range(mc)],0)
    prob = ps.mean(0)[0,0].cpu().numpy(); unc = ps.std(0)[0,0].cpu().numpy()
    conf = float((prob[prob>0.5]).mean()) if (prob>0.5).any() else float(1-prob.mean())
    return g, prob, unc, conf

# pick one val image + its class from val-text.json
import json
val_text = json.load(open(f'{DATA_ROOT}/val-text.json'))
entry = val_text[0]
name = entry.get('image_name'); cls = entry.get('class_name')
g, prob, unc, conf = predict_with_conf(f'{DATA_ROOT}/val/{name}')
print('image', name, '| class', cls, '| confidence', round(conf,3))

## 3. Deterministic attributes + LLM diagnosis

In [ ]:
from src.llm.diagnose import diagnose

attrs = extract_attributes(g, prob)
print('deterministic attributes:'); print(json.dumps({k:v for k,v in attrs.items() if k!='_raw'}, indent=2))

diag = diagnose(cls, t1[cls], attrs, confidence=conf, uncertainty=float(unc.mean()), provider=PROVIDER)
print('\nLLM diagnostic note:')
for k in ('likely_cause','severity','recommended_action','summary'):
    print(f'  {k}: {diag.get(k)}')
print('  valid:', diag.get('_valid'))

## 4. Automated evaluation over a sample (validity / grounding / faithfulness)

Runs the diagnostic on ~25 val images and reports the three defensible metrics.

In [ ]:
from src.llm.evaluate_llm import evaluate_batch
import random
random.seed(0)
sample = random.sample(val_text, 25)

records = []
for e in sample:
    nm, cl = e.get('image_name'), e.get('class_name')
    try:
        gg, pp, uu, cc = predict_with_conf(f'{DATA_ROOT}/val/{nm}')
        aa = extract_attributes(gg, pp)
        dd = diagnose(cl, t1[cl], aa, confidence=cc, uncertainty=float(uu.mean()), provider=PROVIDER)
        records.append({'diag': dd, 't1_cause': t1[cl], 'attributes': aa})
    except Exception as ex:
        print('skip', nm, ex)

metrics = evaluate_batch(records)
print('\nLLM DIAGNOSTIC EVAL:', json.dumps(metrics, indent=2))
os.makedirs('results/llm', exist_ok=True)
json.dump(metrics, open('results/llm/eval_metrics.json','w'), indent=2)
json.dump([r['diag'] for r in records], open('results/llm/sample_diagnoses.json','w'), indent=2)
print('saved -> results/llm/eval_metrics.json + sample_diagnoses.json')